# Olist Dataset Ingestion

## Purpose

Download the Olist Brazilian E-Commerce dataset directly from Kaggle and store the extracted CSV files inside the Microsoft Fabric Lakehouse landing zone.

## Source

- Dataset: Olist Brazalian E-Commerce Public Dataset
- Provider: OList
- Platform: Kaggle
- Dataset Link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Security

The Kaggle API credential is used temporarily and deleted after successful ingestion.


In [1]:
%pip install -q kaggle

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
import os
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 9, Finished, Available, Finished, False)

In [3]:
LAKEHOUSE_ROOT = Path("/lakehouse/default/")

CREDENTIAL_SOURCE = (
    LAKEHOUSE_ROOT
    / "Files"
    / "reference"
    / "private"
    / "kaggle.json")

KAGGLE_CONFIG_DIR = Path.home() / ".kaggle"
CREDENTIAL_DESTINATION = KAGGLE_CONFIG_DIR / "Kaggle.json"

LANDING_ROOT = (
    LAKEHOUSE_ROOT
    / "Files"
    / "landing"
    / "olist"
)

SOURCE_CSV_DIR = LANDING_ROOT / "source_csv"
DOWNLOAD_DIR = LANDING_ROOT / "_download"

DATASET_IDENTIFIER = "olistbr/brazilian-ecommerce"

print("Credential source:", CREDENTIAL_SOURCE)
print("Landing root:", LANDING_ROOT)
print("CSV destination:", SOURCE_CSV_DIR)

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 10, Finished, Available, Finished, False)

Credential source: /lakehouse/default/Files/reference/private/kaggle.json
Landing root: /lakehouse/default/Files/landing/olist
CSV destination: /lakehouse/default/Files/landing/olist/source_csv


In [4]:
if not CREDENTIAL_SOURCE.exists():
    raise FileNotFoundError(
        "kaggle.json was not found in "
        "Files/reference/private/. Upload it before continuing"
    )

print("Credential file found")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 11, Finished, Available, Finished, False)

Credential file found


In [5]:
KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    CREDENTIAL_SOURCE,
    CREDENTIAL_DESTINATION
)

os.chmod(CREDENTIAL_DESTINATION, 0o600)

os.environ["KAGGLE_CONGIG_DIR"] = str(KAGGLE_CONFIG_DIR)

print("Temporary Kaggle credential configured")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 12, Finished, Available, Finished, False)

Temporary Kaggle credential configured


In [6]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CSV_DIR.mkdir(parents=True, exist_ok=True)

print("Download Directory: ", DOWNLOAD_DIR)
print("CSV Directory: ", SOURCE_CSV_DIR)

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 13, Finished, Available, Finished, False)

Download Directory:  /lakehouse/default/Files/landing/olist/_download
CSV Directory:  /lakehouse/default/Files/landing/olist/source_csv


In [16]:
import subprocess

command = [
    sys.executable,
    "-m",
    "kaggle",
    "datasets",
    "download",
    "-d",
    DATASET_IDENTIFIER,
    "-p",
    str(DOWNLOAD_DIR),
    "--force"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Kaggle dataset download failed.")

print("Dataset download completed successfully.")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 23, Finished, Available, Finished, False)

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0


Dataset download completed successfully.


In [17]:
zip_files = list(DOWNLOAD_DIR.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError(
        "No ZIP file was found after the Kaggle download."
    )

for file in zip_files:
    print(file.name, file.stat().st_size, "bytes")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 24, Finished, Available, Finished, False)

brazilian-ecommerce.zip 44717580 bytes


In [18]:
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path,"r") as archive:
        archive.extractall(SOURCE_CSV_DIR)

print("Dataset extracted successfully")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 25, Finished, Available, Finished, False)

Dataset extracted successfully


In [21]:
EXPECTED_FILES = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

actual_files = {
    file.name
    for file in SOURCE_CSV_DIR.glob("*.csv")
}

missing_files = EXPECTED_FILES - actual_files
unexpected_files = actual_files - EXPECTED_FILES

print("Expected files:", len(EXPECTED_FILES))
print("Actual CSV files:", len(actual_files))
print("Missing files:", missing_files)
print("Unexpected files:", unexpected_files)

if missing_files:
    raise ValueError(
        f"Ingestion validation failed. Missing files: {missing_files}"
    )

print("All expected Olist files are present.")

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 28, Finished, Available, Finished, False)

Expected files: 9
Actual CSV files: 9
Missing files: set()
Unexpected files: set()
All expected Olist files are present.


In [22]:
source_inventory = []

for file in sorted(SOURCE_CSV_DIR.glob("*.csv")):
    source_inventory.append(
        {
            "filename": file.name,
            "size_bytes": file.stat().st_size,
            "size_mb": round(file.stat().st_size / (1024*1024),3),
            "ingested_at_utc": datetime.now(timezone.utc).isoformat()
        }
    )

source_inventory

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 29, Finished, Available, Finished, False)

[{'filename': 'olist_customers_dataset.csv',
  'size_bytes': 9033957,
  'size_mb': 8.615,
  'ingested_at_utc': '2026-07-26T17:36:12.525865+00:00'},
 {'filename': 'olist_geolocation_dataset.csv',
  'size_bytes': 61273883,
  'size_mb': 58.435,
  'ingested_at_utc': '2026-07-26T17:36:12.525974+00:00'},
 {'filename': 'olist_order_items_dataset.csv',
  'size_bytes': 15438671,
  'size_mb': 14.723,
  'ingested_at_utc': '2026-07-26T17:36:12.526066+00:00'},
 {'filename': 'olist_order_payments_dataset.csv',
  'size_bytes': 5777138,
  'size_mb': 5.51,
  'ingested_at_utc': '2026-07-26T17:36:12.526147+00:00'},
 {'filename': 'olist_order_reviews_dataset.csv',
  'size_bytes': 14451670,
  'size_mb': 13.782,
  'ingested_at_utc': '2026-07-26T17:36:12.526229+00:00'},
 {'filename': 'olist_orders_dataset.csv',
  'size_bytes': 17654914,
  'size_mb': 16.837,
  'ingested_at_utc': '2026-07-26T17:36:12.526314+00:00'},
 {'filename': 'olist_products_dataset.csv',
  'size_bytes': 2379446,
  'size_mb': 2.269,
  'ing

In [23]:
import pandas as pd

inventory_df = pd.DataFrame(source_inventory)

display(inventory_df)

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0cc0b81-5b06-43ca-bfdd-50c550a7a837)

In [24]:
deleted_paths = []

for credential_path in [
    CREDENTIAL_DESTINATION,
    CREDENTIAL_SOURCE
]:
    if credential_path.exists():
        credential_path.unlink()
        deleted_paths.append(str(credential_path))

print("Deleted credential files:")
for path in deleted_paths:
    print(path)

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 31, Finished, Available, Finished, False)

Deleted credential files:
/home/trusted-service-user/.kaggle/Kaggle.json
/lakehouse/default/Files/reference/private/kaggle.json


In [26]:
for zip_path in zip_files:
    if zip_path.exists():
        zip_path.unlink()
        print("Deleted:", zip_path.name)

StatementMeta(, d6f2c4a9-4373-4257-ab56-3a6e09456832, 33, Finished, Available, Finished, False)

Deleted: brazilian-ecommerce.zip
